### What we are going to implement:

> Dense Layer

> ReLU

> Softmax

> Cross-Entropy

> Backpropagation

> Adam optimizer

> Bias correction

> Batch training

This is basically what PyTorch does internally.

m = β1*m + (1-β1)*grad
v = β2*v + (1-β2)*(grad²)

m_corrected = m / (1 - β1^t)
v_corrected = v / (1 - β2^t)

W = W - lr * m_corrected / (sqrt(v_corrected) + ε)

Where:

β1 = 0.9

β2 = 0.999

ε = 1e-8

t = time step


In [1]:
import numpy as np

np.random.seed(0)

# ---------------------------
# DATA (3 samples, 3 classes)
# ---------------------------
X = np.array([
    [1.0, 2.0, 3.0, 2.5],
    [2.0, 5.0, -1.0, 2.0],
    [-1.5, 2.7, 3.3, -0.8]
])

y_true = np.array([
    [0, 0, 1],
    [1, 0, 0],
    [0, 1, 0]
])

# ---------------------------
# INITIALIZE PARAMETERS
# ---------------------------
W1 = 0.01 * np.random.randn(4, 3)
b1 = np.zeros((1, 3))

W2 = 0.01 * np.random.randn(3, 3)
b2 = np.zeros((1, 3))

# ---------------------------
# ADAM VARIABLES
# ---------------------------
mW1 = np.zeros_like(W1)
vW1 = np.zeros_like(W1)
mb1 = np.zeros_like(b1)
vb1 = np.zeros_like(b1)

mW2 = np.zeros_like(W2)
vW2 = np.zeros_like(W2)
mb2 = np.zeros_like(b2)
vb2 = np.zeros_like(b2)

learning_rate = 0.01
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
epochs = 1000

# ---------------------------
# TRAINING LOOP
# ---------------------------
for epoch in range(1, epochs + 1):

    # -------- FORWARD PASS --------
    z1 = np.dot(X, W1) + b1
    a1 = np.maximum(0, z1)   # ReLU

    z2 = np.dot(a1, W2) + b2

    exp_vals = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
    y_pred = exp_vals / np.sum(exp_vals, axis=1, keepdims=True)

    # Cross-entropy loss
    loss = -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))

    # -------- BACKWARD PASS --------
    m = X.shape[0]

    dz2 = (y_pred - y_true) / m
    dW2 = np.dot(a1.T, dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = np.dot(dz2, W2.T)
    dz1 = da1 * (z1 > 0)

    dW1 = np.dot(X.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # -------- ADAM UPDATE --------
    # ---- Layer 2 ----
    mW2 = beta1 * mW2 + (1 - beta1) * dW2
    vW2 = beta2 * vW2 + (1 - beta2) * (dW2 ** 2)

    mb2 = beta1 * mb2 + (1 - beta1) * db2
    vb2 = beta2 * vb2 + (1 - beta2) * (db2 ** 2)

    mW2_corr = mW2 / (1 - beta1 ** epoch)
    vW2_corr = vW2 / (1 - beta2 ** epoch)

    mb2_corr = mb2 / (1 - beta1 ** epoch)
    vb2_corr = vb2 / (1 - beta2 ** epoch)

    W2 -= learning_rate * mW2_corr / (np.sqrt(vW2_corr) + epsilon)
    b2 -= learning_rate * mb2_corr / (np.sqrt(vb2_corr) + epsilon)

    # ---- Layer 1 ----
    mW1 = beta1 * mW1 + (1 - beta1) * dW1
    vW1 = beta2 * vW1 + (1 - beta2) * (dW1 ** 2)

    mb1 = beta1 * mb1 + (1 - beta1) * db1
    vb1 = beta2 * vb1 + (1 - beta2) * (db1 ** 2)

    mW1_corr = mW1 / (1 - beta1 ** epoch)
    vW1_corr = vW1 / (1 - beta2 ** epoch)

    mb1_corr = mb1 / (1 - beta1 ** epoch)
    vb1_corr = vb1 / (1 - beta2 ** epoch)

    W1 -= learning_rate * mW1_corr / (np.sqrt(vW1_corr) + epsilon)
    b1 -= learning_rate * mb1_corr / (np.sqrt(vb1_corr) + epsilon)

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

print("\nFinal Predictions:")
print(np.round(y_pred, 3))


Epoch 200, Loss: 0.0416
Epoch 400, Loss: 0.0139
Epoch 600, Loss: 0.0073
Epoch 800, Loss: 0.0046
Epoch 1000, Loss: 0.0032

Final Predictions:
[[0.    0.    1.   ]
 [1.    0.    0.   ]
 [0.005 0.991 0.005]]
